## Intelligent University Exam Scheduling System 

In [28]:
import random

courses = {
    'C1': {'students': 40, 'type': 'Any'}, 'C2': {'students': 30, 'type': 'Any'},
    'C3': {'students': 25, 'type': 'Lab'}, 'C4': {'students': 35, 'type': 'Any'}
}
rooms = {
    'R1': {'cap': 50, 'type': 'Regular'}, 'R2': {'cap': 35, 'type': 'Regular'},
    'Lab1': {'cap': 30, 'type': 'Lab'}
}
timeslots = ['T1', 'T2', 'T3']
days = {'T1': 'Day 1', 'T2': 'Day 1', 'T3': 'Day 2'}
conflicts = [('C1', 'C2'), ('C2', 'C4')]

course_keys = list(courses.keys())
room_keys = list(rooms.keys())

def get_cost(schedule):
    penalty = 0
    used_rooms = set()

    for course, (room, time) in schedule.items():
        if rooms[room]['cap'] < courses[course]['students']: penalty += 1000
        if courses[course]['type'] == 'Lab' and rooms[room]['type'] != 'Lab': penalty += 1000
        if (room, time) in used_rooms: penalty += 1000
        used_rooms.add((room, time))
        
        penalty += (rooms[room]['cap'] - courses[course]['students'])

    for c1, c2 in conflicts:
        r1, t1 = schedule[c1]
        r2, t2 = schedule[c2]
        if t1 == t2: penalty += 1000
        elif days[t1] == days[t2]: penalty += 50

    return penalty

def random_schedule():
    return {c: (random.choice(room_keys), random.choice(timeslots)) for c in course_keys}

def crossover(parent1, parent2):
    child = {}
    for course in course_keys:
        child[course] = parent1[course] if random.random() < 0.5 else parent2[course]
    return child

def mutate(schedule):
    if random.random() < 0.2:
        course = random.choice(course_keys)
        schedule[course] = (random.choice(room_keys), random.choice(timeslots))
    return schedule

population = [random_schedule() for _ in range(50)]

for generation in range(200):
    population.sort(key=get_cost)
    
    if get_cost(population[0]) < 1000:
        break
        
    top_half = population[:25]
    
    next_generation = []
    while len(next_generation) < 50:
        p1 = random.choice(top_half)
        p2 = random.choice(top_half)
        
        child = crossover(p1, p2)
        child = mutate(child)
        next_generation.append(child)
        
    population = next_generation

# Print Result
best = population[0]
print("****************** Final Schedule ******************")
for course, (room, time) in best.items():
    print(f"{course} -> Room: {room} | Time: {time} ({days[time]})")
print(f"Total Penalty Cost: {get_cost(best)}")

****************** Final Schedule ******************
C1 -> Room: R1 | Time: T2 (Day 1)
C2 -> Room: Lab1 | Time: T3 (Day 2)
C3 -> Room: Lab1 | Time: T2 (Day 1)
C4 -> Room: R2 | Time: T1 (Day 1)
Total Penalty Cost: 15
